In [ ]:
import struct
import numpy as np

def convert_oscilloscope_data(filename):
    """
    Convierte datos binarios de osciloscopio con formato específico:
    - Header ASCII: "#48008" (6 bytes)
    - Intervalo de tiempo IEEE754 16-bit (2 bytes, little-endian)
    - Bloque no usado (3 bytes)
    - 8000 muestras de 2 bytes cada una (MSB first, big-endian por muestra)
    """
    
    with open(filename, 'rb') as file:
        # Leer header ASCII
        header = file.read(6).decode('ascii')
        print(f"Header: {header}")
        
        # Leer intervalo de tiempo (IEEE754 16-bit, little-endian)
        time_interval_bytes = file.read(2)
        time_interval = ieee754_16bit_to_float(time_interval_bytes, little_endian=True)
        print(f"Intervalo de tiempo: {time_interval}")
        
        # Saltar bloque no usado (3 bytes)
        file.read(3)
        
        # Leer 8000 muestras de forma de onda
        waveform_data = []
        for i in range(8000):
            # Leer 2 bytes por muestra (MSB first = big-endian)
            sample_bytes = file.read(2)
            if len(sample_bytes) < 2:
                break
            
            # Convertir a IEEE754 16-bit
            sample_value = ieee754_16bit_to_float(sample_bytes, little_endian=False)
            waveform_data.append(sample_value)
        
        # Crear array de tiempo
        time_array = np.arange(len(waveform_data)) * time_interval
        
        return np.array(time_array), np.array(waveform_data)

def ieee754_16bit_to_float(bytes_data, little_endian=True):
    """
    Convierte 2 bytes a float usando IEEE754 half-precision (16-bit)
    Formato: 1 bit signo + 5 bits exponente + 10 bits mantisa
    """
    
    if little_endian:
        # Little-endian: byte menos significativo primero
        value = (bytes_data[1] << 8) | bytes_data[0]
    else:
        # Big-endian: byte más significativo primero
        value = (bytes_data[0] << 8) | bytes_data[1]
    
    # Extraer componentes IEEE754 16-bit
    sign = (value >> 15) & 0x1
    exponent = (value >> 10) & 0x1F
    mantissa = value & 0x3FF
    
    # Casos especiales
    if exponent == 0:
        if mantissa == 0:
            result = 0.0
        else:
            # Número denormalizado
            result = (mantissa / 1024.0) * (2 ** -14)
    elif exponent == 31:
        if mantissa == 0:
            result = float('inf')
        else:
            result = float('nan')
    else:
        # Número normalizado
        result = (1 + mantissa / 1024.0) * (2 ** (exponent - 15))
    
    return -result if sign else result

def test_with_sample_data():
    """
    Prueba con los datos de muestra proporcionados
    """
    # Datos de muestra en binario (convertir a bytes)
    sample_binary = [
        0b00100011, 0b00110100, 0b00111000, 0b00110000, 0b00110000, 
        0b00111000, 0b00110101, 0b11100010, 0b10010110, 0b10010010, 
        0b00110111, 0b11100010, 0b10010110
    ]
    
    sample_bytes = bytes(sample_binary)
    
    print("Análisis de datos de muestra:")
    print(f"Total bytes: {len(sample_bytes)}")
    
    # Header
    header = sample_bytes[:6].decode('ascii')
    print(f"Header: '{header}'")
    
    # Intervalo de tiempo
    time_bytes = sample_bytes[6:8]
    time_interval = ieee754_16bit_to_float(time_bytes, little_endian=False)
    print(f"Intervalo de tiempo: {time_interval}")

    # Datos restantes (después del bloque de 3 bytes)
    remaining_bytes = len(sample_bytes) - 11  # 6 header + 2 time + 3 unused
    print(f"Bytes restantes para muestras: {remaining_bytes}")

# Ejemplo de uso
if __name__ == "__main__":
    # PRUEBA 1:
    test_with_sample_data()

    # PRUEBA 2:
    # 0b0011101111111111 = 0.99951172
    sample_binary = [ 0b00111011, 0b11111111 ]
    time_bytes = bytes(sample_binary)
    time_interval_float = ieee754_16bit_to_float(time_bytes, little_endian=False)
    print(f"Intervalo de tiempo 2: {time_interval_float}")

Análisis de datos de muestra:
Total bytes: 13
Header: '#48008'
Intervalo de tiempo: 0.36767578125
Bytes restantes para muestras: 2
Intervalo de tiempo 2: 0.99951171875
